# Leximin-optimal assignment for the doctor/hospital problem

Leximin directly targets that: among all valid doctor-to-hospital assignments, it picks the one that makes the worst-off doctor's rank as good as possible, then (subject to that) makes the second-worst-off doctor's rank as good as possible, and so on.

**Implementation ("peel the forced-worst" method):** repeatedly
1. find the best (smallest) worst-case rank `M` achievable for the doctors still unassigned, via bipartite matching feasibility checks,
2. among assignments that achieve `M`, find one using the fewest possible doctors stuck at `M`, via a min-cost bipartite matching,
3. permanently fix whichever doctors ended up at `M` this round (they can't do better without hurting the bound for someone else) and remove them and their hospital from the pool, then repeat on what's left.

Each hospital can now take more than one doctor: pass a `capacities` dict (hospital -> seat count). Internally, a hospital with `c` remaining seats is expanded into `c` identical-cost "slot" columns before each min-cost matching, so the same Hungarian-algorithm machinery handles multi-seat hospitals without any other change to the algorithm.



In [1]:
from __future__ import annotations

import numpy as np
from scipy.optimize import linear_sum_assignment

In [2]:
def _rank_lookup(preferences: dict[str, list[str]]) -> dict[str, dict[str, int]]:
    return {
        doctor: {hospital: rank for rank, hospital in enumerate(hospitals, start=1)}
        for doctor, hospitals in preferences.items()
    }

In [3]:
def _min_cost_matching(
    doctors: list[str],
    hospitals: list[str],
    capacities: dict[str, int],
    cost: dict[tuple[str, str], float],
) -> list[tuple[str, str]] | None:
    """Solve a min-cost matching where each doctor gets exactly one
    hospital and each hospital is used at most `capacities[hospital]`
    times, by expanding each hospital into that many identical-cost
    "slot" columns and solving a standard assignment problem. Missing
    (doctor, hospital) pairs are disallowed. Returns the matched
    (doctor, hospital) pairs, or None if no complete assignment is
    possible.
    """
    columns = [h for h in hospitals for _ in range(capacities[h])]
    matrix = np.full((len(doctors), len(columns)), np.inf)
    for i, doctor in enumerate(doctors):
        for j, hospital in enumerate(columns):
            if (doctor, hospital) in cost:
                matrix[i, j] = cost[(doctor, hospital)]

    try:
        row_ind, col_ind = linear_sum_assignment(matrix)
    except ValueError:
        return None

    return [(doctors[r], columns[c]) for r, c in zip(row_ind, col_ind)]

In [4]:
def leximin_assignment(
    preferences: dict[str, list[str]],
    capacities: dict[str, int],
    verbose: bool = False,
) -> dict[str, str]:
    """Find the leximin-optimal doctor -> hospital assignment.

    Args:
        preferences: doctor -> hospitals ranked most-to-least preferred.
            Every doctor must rank enough hospitals that a complete
            assignment is actually possible.
        capacities: hospital -> number of seats it has. Total capacity
            must be at least the number of doctors.
        verbose: if True, print each round's chosen worst-case rank and
            which doctors got locked in at it.

    Returns:
        doctor -> hospital, the final assignment.
    """
    ranks = _rank_lookup(preferences)
    if sum(capacities.values()) < len(preferences):
        raise ValueError("total hospital capacity is less than the number of doctors")

    remaining_doctors = list(preferences.keys())
    remaining_capacity = dict(capacities)
    assignment: dict[str, str] = {}
    round_num = 0

    while remaining_doctors:
        round_num += 1
        remaining_hospitals = [h for h, c in remaining_capacity.items() if c > 0]

        candidate_ranks = sorted(
            {
                ranks[d][h]
                for d in remaining_doctors
                for h in remaining_hospitals
                if h in ranks[d]
            }
        )

        best_m = None
        for m in candidate_ranks:
            cost = {
                (d, h): 0
                for d in remaining_doctors
                for h in remaining_hospitals
                if h in ranks[d] and ranks[d][h] <= m
            }
            if _min_cost_matching(remaining_doctors, remaining_hospitals, remaining_capacity, cost) is not None:
                best_m = m
                break

        if best_m is None:
            raise ValueError(
                "no complete assignment is possible -- some doctor's "
                "preference list doesn't cover enough hospitals"
            )

        cost = {
            (d, h): (1 if ranks[d][h] == best_m else 0)
            for d in remaining_doctors
            for h in remaining_hospitals
            if h in ranks[d] and ranks[d][h] <= best_m
        }
        pairs = _min_cost_matching(remaining_doctors, remaining_hospitals, remaining_capacity, cost)
        assert pairs is not None  # feasible by construction (best_m worked above)

        settled = [(d, h) for d, h in pairs if ranks[d][h] == best_m]

        if verbose:
            print(f"Round {round_num}: worst achievable rank = {best_m}")
            for doctor, hospital in settled:
                print(f"  locking in {doctor} -> {hospital} (rank {best_m})")

        for doctor, hospital in settled:
            assignment[doctor] = hospital
            remaining_doctors.remove(doctor)
            remaining_capacity[hospital] -= 1

    return assignment

In [5]:
def preferences_from_ranks(
    ranks: dict[str, dict[str, int]]
) -> dict[str, list[str]]:
    """Convert doctor -> {hospital: rank} (1 = most preferred) into
    doctor -> [hospitals ordered most-to-least preferred].
    """
    return {
        doctor: [h for h, _ in sorted(hospital_ranks.items(), key=lambda kv: kv[1])]
        for doctor, hospital_ranks in ranks.items()
    }

## Example

A case with real conflict: Dr. Alvarez, Dr. Chen, and Dr. Diallo all rank General as their #1 choice, so at least one of them can't get it. Leximin picks whichever assignment makes that unlucky doctor's outcome as good as possible, rather than leaving it to luck.

In [6]:
ranks = {
    "Dr. Alvarez": {"General": 1, "St. Mary": 2, "County": 3, "Metro": 4},
    "Dr. Chen":    {"General": 1, "County": 2, "St. Mary": 3, "Metro": 4},
    "Dr. Diallo":  {"General": 1, "Metro": 2, "County": 3, "St. Mary": 4},
    "Dr. Kapoor":  {"Metro": 1, "St. Mary": 2, "County": 3, "General": 4},
}
preferences = preferences_from_ranks(ranks)
capacities = {"General": 1, "St. Mary": 1, "County": 1, "Metro": 1}  # one seat each, as before

assignment = leximin_assignment(preferences, capacities, verbose=True)

print("\nFinal assignment:")
for doctor, hospital in assignment.items():
    rank = ranks[doctor][hospital]
    print(f"  {doctor} -> {hospital} (was their #{rank} choice)")

Round 1: worst achievable rank = 2
  locking in Dr. Alvarez -> St. Mary (rank 2)
  locking in Dr. Chen -> County (rank 2)
Round 2: worst achievable rank = 1
  locking in Dr. Diallo -> General (rank 1)
  locking in Dr. Kapoor -> Metro (rank 1)

Final assignment:
  Dr. Alvarez -> St. Mary (was their #2 choice)
  Dr. Chen -> County (was their #2 choice)
  Dr. Diallo -> General (was their #1 choice)
  Dr. Kapoor -> Metro (was their #1 choice)


## Baseline comparison: Random Serial Dictatorship (RSD)

RSD : draw a random lottery order for the doctors, then one at a time each doctor simply takes their favorite hospital that's still open. 

What RSD does *not* do is bound the worst outcome. Whichever doctor happens to be unlucky in the lottery order can get stuck with a much worse choice: result below shows the achieved worst-case rank swinging between draws.

In [7]:
import random


def random_order(doctors: list[str], seed: int | None = None) -> list[str]:
    """Draw a random pick order (the lottery step of Random Serial Dictatorship)."""
    rng = random.Random(seed)
    order = list(doctors)
    rng.shuffle(order)
    return order


def serial_dictatorship(
    preferences: dict[str, list[str]],
    order: list[str],
    capacities: dict[str, int],
) -> dict[str, str]:
    """Doctors pick, one at a time in `order`, their favorite hospital that still has an open seat."""
    remaining_capacity = dict(capacities)
    assignment: dict[str, str] = {}
    for doctor in order:
        favorite = next(h for h in preferences[doctor] if remaining_capacity.get(h, 0) > 0)
        assignment[doctor] = favorite
        remaining_capacity[favorite] -= 1
    return assignment

In [8]:
leximin_worst = max(ranks[d][h] for d, h in assignment.items())
print(f"Leximin worst-case rank (guaranteed): {leximin_worst}\n")

print("Random Serial Dictatorship across 5 independent lottery draws:")
for seed in range(5):
    order = random_order(list(preferences), seed=seed)
    rsd_assignment = serial_dictatorship(preferences, order, capacities)
    achieved = {d: ranks[d][h] for d, h in rsd_assignment.items()}
    worst = max(achieved.values())
    flag = "worse than leximin" if worst > leximin_worst else "matches leximin"
    print(f"  seed {seed}: order={order}")
    print(f"    ranks achieved={achieved} -> worst={worst} ({flag})")

Leximin worst-case rank (guaranteed): 2

Random Serial Dictatorship across 5 independent lottery draws:
  seed 0: order=['Dr. Diallo', 'Dr. Alvarez', 'Dr. Chen', 'Dr. Kapoor']
    ranks achieved={'Dr. Diallo': 1, 'Dr. Alvarez': 2, 'Dr. Chen': 2, 'Dr. Kapoor': 1} -> worst=2 (matches leximin)
  seed 1: order=['Dr. Kapoor', 'Dr. Alvarez', 'Dr. Diallo', 'Dr. Chen']
    ranks achieved={'Dr. Kapoor': 1, 'Dr. Alvarez': 1, 'Dr. Diallo': 3, 'Dr. Chen': 3} -> worst=3 (worse than leximin)
  seed 2: order=['Dr. Chen', 'Dr. Diallo', 'Dr. Kapoor', 'Dr. Alvarez']
    ranks achieved={'Dr. Chen': 1, 'Dr. Diallo': 2, 'Dr. Kapoor': 2, 'Dr. Alvarez': 3} -> worst=3 (worse than leximin)
  seed 3: order=['Dr. Kapoor', 'Dr. Alvarez', 'Dr. Diallo', 'Dr. Chen']
    ranks achieved={'Dr. Kapoor': 1, 'Dr. Alvarez': 1, 'Dr. Diallo': 3, 'Dr. Chen': 3} -> worst=3 (worse than leximin)
  seed 4: order=['Dr. Diallo', 'Dr. Alvarez', 'Dr. Kapoor', 'Dr. Chen']
    ranks achieved={'Dr. Diallo': 1, 'Dr. Alvarez': 2, 'Dr. Kap

## Multi-seat hospitals

A bigger example where each hospital can take more than one doctor: General and County each have 2 seats, Metro and St. Mary have 1 each -- 6 seats total for 6 doctors. Three doctors (D1, D2, D3) all rank General #1, and two (D4, D5) both rank County #1, so there's still contention, just softened by the extra seats: only one doctor per contested hospital has to be displaced instead of all-but-one.

In [9]:
cap_ranks = {
    "D1": {"General": 1, "County": 2, "Metro": 3, "St. Mary": 4},
    "D2": {"General": 1, "St. Mary": 2, "County": 3, "Metro": 4},
    "D3": {"General": 1, "Metro": 2, "St. Mary": 3, "County": 4},
    "D4": {"County": 1, "General": 2, "St. Mary": 3, "Metro": 4},
    "D5": {"County": 1, "Metro": 2, "General": 3, "St. Mary": 4},
    "D6": {"Metro": 1, "St. Mary": 2, "General": 3, "County": 4},
}
cap_preferences = preferences_from_ranks(cap_ranks)
cap_capacities = {"General": 2, "County": 2, "Metro": 1, "St. Mary": 1}

cap_assignment = leximin_assignment(cap_preferences, cap_capacities, verbose=True)

print("\nFinal assignment:")
for doctor, hospital in cap_assignment.items():
    rank = cap_ranks[doctor][hospital]
    print(f"  {doctor} -> {hospital} (was their #{rank} choice)")

Round 1: worst achievable rank = 2
  locking in D2 -> St. Mary (rank 2)
Round 2: worst achievable rank = 1
  locking in D1 -> General (rank 1)
  locking in D3 -> General (rank 1)
  locking in D4 -> County (rank 1)
  locking in D5 -> County (rank 1)
  locking in D6 -> Metro (rank 1)

Final assignment:
  D2 -> St. Mary (was their #2 choice)
  D1 -> General (was their #1 choice)
  D3 -> General (was their #1 choice)
  D4 -> County (was their #1 choice)
  D5 -> County (was their #1 choice)
  D6 -> Metro (was their #1 choice)


With two seats at General, only one of the three doctors who wanted it has to settle for a worse choice -- leximin's guaranteed worst-case rank drops accordingly. Comparing against RSD on the same capacities shows the same pattern as before: RSD sometimes matches leximin's bound by luck, but can also land a doctor much further down their list.

In [10]:
cap_leximin_worst = max(cap_ranks[d][h] for d, h in cap_assignment.items())
print(f"Leximin worst-case rank (guaranteed): {cap_leximin_worst}\n")

print("Random Serial Dictatorship across 6 independent lottery draws:")
for seed in range(6):
    order = random_order(list(cap_preferences), seed=seed)
    rsd_assignment = serial_dictatorship(cap_preferences, order, cap_capacities)
    achieved = {d: cap_ranks[d][h] for d, h in rsd_assignment.items()}
    worst = max(achieved.values())
    flag = "worse than leximin" if worst > cap_leximin_worst else "matches leximin"
    print(f"  seed {seed}: worst={worst} ({flag})  ranks achieved={achieved}")

Leximin worst-case rank (guaranteed): 2

Random Serial Dictatorship across 6 independent lottery draws:
  seed 0: worst=3 (worse than leximin)  ranks achieved={'D5': 1, 'D3': 1, 'D2': 1, 'D1': 2, 'D6': 1, 'D4': 3}
  seed 1: worst=2 (matches leximin)  ranks achieved={'D3': 1, 'D4': 1, 'D6': 1, 'D1': 1, 'D5': 1, 'D2': 2}
  seed 2: worst=4 (worse than leximin)  ranks achieved={'D3': 1, 'D4': 1, 'D2': 1, 'D5': 1, 'D6': 1, 'D1': 4}
  seed 3: worst=2 (matches leximin)  ranks achieved={'D1': 1, 'D3': 1, 'D4': 1, 'D6': 1, 'D5': 1, 'D2': 2}
  seed 4: worst=2 (matches leximin)  ranks achieved={'D4': 1, 'D6': 1, 'D5': 1, 'D1': 1, 'D3': 1, 'D2': 2}
  seed 5: worst=3 (worse than leximin)  ranks achieved={'D2': 1, 'D1': 1, 'D4': 1, 'D6': 1, 'D3': 3, 'D5': 1}
